## Acquisition automatique des données brutes

Projet :

**Objectif de ce notebook**

Télécharger, extraire et organiser de manière **100 % reproductible** les trois sources de données brutes nécessaires au pipeline :

1. OpenStreetMap (Geofabrik)  
2. Hydrographie CartoMaroc  
3. DEM Grand Casablanca 30 m  

**CRS cible** : `EPSG:26191` (Nord Maroc)  
**Date** : 21 octobre 2026

## Structure des dossiers
```text
donnees_brutes/
├── osm/                          ← Shapefiles OpenStreetMap (Geofabrik)
├── source2/
│   ├── source2_donnees.rar
│   └── extracted/
│       └── hydro-maroc(CartoMaroc)/
├── dem/
│   └── DEM_Grand_Casablanca_30m.tif
└── processed/                    ← Versions reprojetées (EPSG:26191)

Installation des bibliothèques

In [ ]:
pip install gdown rarfile tqdm certifi rasterio geopandas -q
print("✓ Packages installés")

## 1. Source OpenStreetMap – Geofabrik

**Source** : https://download.geofabrik.de/africa/morocco-latest-free.shp.zip  
**Format** : Shapefiles thématiques  
**Contenu** : routes, bâtiments, occupation du sol, cours d’eau, POI, etc.

In [ ]:
# =============================================================================
# BLOC 1 : TÉLÉCHARGEMENT ET EXTRACTION DES DONNÉES OSM (Geofabrik)
# Objectif : Récupérer automatiquement l'extrait shapefile du Maroc
# Source   : https://download.geofabrik.de
# =============================================================================

from pathlib import Path
import requests
import certifi
import os
from tqdm.auto import tqdm   # barre de progression propre
import zipfile               # nécessaire pour la vérification et l'extraction

# -----------------------------------------------------------------------------
# Correction du problème de certificat SSL (conflit fréquent avec PostgreSQL)
# -----------------------------------------------------------------------------
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# -----------------------------------------------------------------------------
# Configuration des chemins et de l'URL
# -----------------------------------------------------------------------------
GEOFABRIK_URL = "https://download.geofabrik.de/africa/morocco-latest-free.shp.zip"
GEOFABRIK_ZIP = Path("donnees_brutes/osm/morocco-latest-free.shp.zip")
OSM_DIR = Path("donnees_brutes/osm")          # dossier de destination des shapefiles

# -----------------------------------------------------------------------------
# 1. Nettoyage : suppression d'un éventuel fichier corrompu précédent
# -----------------------------------------------------------------------------
if GEOFABRIK_ZIP.exists():
    print("Suppression du fichier corrompu...")
    GEOFABRIK_ZIP.unlink()

# -----------------------------------------------------------------------------
# 2. Téléchargement robuste avec barre de progression
# -----------------------------------------------------------------------------
print("↓ Nouveau téléchargement Geofabrik...")
response = requests.get(GEOFABRIK_URL, stream=True, verify=certifi.where(), timeout=60)
response.raise_for_status()

total_size = int(response.headers.get("content-length", 0))

with open(GEOFABRIK_ZIP, "wb") as f, tqdm(
    desc="Geofabrik Morocco",
    total=total_size,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=1024*1024):  # 1 Mo
        if chunk:
            f.write(chunk)
            bar.update(len(chunk))

print(f"\n✓ Téléchargement terminé : {GEOFABRIK_ZIP.stat().st_size / (1024*1024):.1f} Mo")

# -----------------------------------------------------------------------------
# 3. Vérification de l'intégrité du fichier ZIP
# -----------------------------------------------------------------------------
print("Vérification du zip...")
with zipfile.ZipFile(GEOFABRIK_ZIP, "r") as z:
    print(f"✓ ZIP valide — {len(z.namelist())} fichiers à l’intérieur")
    print("Exemples :", z.namelist()[:6])

# -----------------------------------------------------------------------------
# 4. Extraction des shapefiles dans le dossier osm/
# -----------------------------------------------------------------------------
print("\n→ Extraction en cours (peut prendre 1-2 minutes)...")
with zipfile.ZipFile(GEOFABRIK_ZIP, "r") as z:
    z.extractall(OSM_DIR)
print("✓ Extraction terminée")

# -----------------------------------------------------------------------------
# 5. Vérification rapide de la présence d'une couche clé
# -----------------------------------------------------------------------------
shp_test = OSM_DIR / "gis_osm_roads_free_1.shp"
print("\nFichier roads présent ?", shp_test.exists())
print("Taille roads.shp :", round(shp_test.stat().st_size / (1024*1024), 1), "Mo" if shp_test.exists() else "")

## 2. Source Hydrographie – CartoMaroc

**Source** : Google Drive (fichier .rar)  
**Contenu** :  
- Oueds (normal + détaillé)  
- Lacs et Dayas  
- Barrages  
- Bassins versants  
- Nappes phréatiques  

Ces couches sont essentielles pour calculer la distance aux cours d’eau et identifier les zones d’accumulation.

In [ ]:

# =============================================================================
# BLOC 2 : TÉLÉCHARGEMENT ET EXTRACTION DES DONNÉES HYDROGRAPHIQUES
# Source   : CartoMaroc (barrages, oueds, lacs, nappes, bassins...)
# Format   : .rar hébergé sur Google Drive
# =============================================================================

from pathlib import Path
import gdown
import subprocess
import shutil
from datetime import datetime

# -----------------------------------------------------------------------------
# Configuration des chemins
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
SOURCE2_DIR = ROOT / "donnees_brutes" / "source2"
SOURCE2_DIR.mkdir(parents=True, exist_ok=True)

FILE_ID = "1w0md7nxYZzoyOhy364yMrdVsUiDpaJ-W"          # ID Google Drive
RAR_PATH = SOURCE2_DIR / "source2_donnees.rar"
EXTRACT_DIR = SOURCE2_DIR / "extracted"

print("=" * 70)
print("ACQUISITION 2ᵉ SOURCE – Hydrographie CartoMaroc")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

# -----------------------------------------------------------------------------
# 1. Téléchargement automatique depuis Google Drive
# -----------------------------------------------------------------------------
if RAR_PATH.exists():
    print(f"✓ Fichier .rar déjà présent ({RAR_PATH.stat().st_size / (1024*1024):.1f} Mo)")
else:
    print("↓ Téléchargement depuis Google Drive...")
    gdown.download(id=FILE_ID, output=str(RAR_PATH), quiet=False)
    print(f"✓ Téléchargé → {RAR_PATH}")

# -----------------------------------------------------------------------------
# 2. Extraction avec WinRAR (méthode la plus fiable sur Windows)
# -----------------------------------------------------------------------------
# Nettoyage du dossier d'extraction précédent
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Recherche de WinRAR
possible_winrar = [
    r"C:\Program Files\WinRAR\WinRAR.exe",
    r"C:\Program Files (x86)\WinRAR\WinRAR.exe",
]
winrar = next((p for p in possible_winrar if Path(p).exists()), None)

if winrar is None:
    print("❌ WinRAR non trouvé. Installez-le ou extrayez manuellement une seule fois.")
else:
    print(f"✓ WinRAR trouvé : {winrar}")
    print("→ Extraction en cours...")
    
    cmd = [winrar, "x", "-y", str(RAR_PATH), str(EXTRACT_DIR) + "\\"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ Extraction réussie avec WinRAR")
    else:
        print("Erreur lors de l'extraction :", result.stderr or result.stdout)

# -----------------------------------------------------------------------------
# 3. Vérification rapide des fichiers extraits
# -----------------------------------------------------------------------------
print("\nFichiers hydrographiques extraits :")
hydro_dir = EXTRACT_DIR / "hydro-maroc(CartoMaroc)"
if hydro_dir.exists():
    for f in sorted(hydro_dir.glob("*.shp")):
        print(f"  - {f.name} ({f.stat().st_size / 1024:.0f} Ko)")
else:
    print("  ⚠ Dossier hydro-maroc(CartoMaroc) non trouvé")

## 3. Source DEM – Grand Casablanca 30 m

**Source** : Google Drive  
**Résolution** : 30 mètres  
**Utilité** : altitude, pente, direction d’écoulement, Topographic Wetness Index (TWI)

In [ ]:
# =============================================================================
# BLOC 3 : TÉLÉCHARGEMENT DU DEM GRAND CASABLANCA 30 m
# Source   : Google Drive
# Résolution : 30 mètres
# =============================================================================

from pathlib import Path
import gdown
from datetime import datetime
import rasterio

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
DEM_DIR = ROOT / "donnees_brutes" / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)

FILE_ID = "15qpvrKivrA5oO6AtYLPgZVen9tUmij6w"           # ID Google Drive du DEM
DEM_PATH = DEM_DIR / "DEM_Grand_Casablanca_30m.tif"

print("=" * 70)
print("ACQUISITION DEM – Grand Casablanca 30 m")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

# -----------------------------------------------------------------------------
# 1. Téléchargement
# -----------------------------------------------------------------------------
if DEM_PATH.exists():
    print(f"✓ DEM déjà présent : {DEM_PATH.name}")
    print(f"  Taille : {DEM_PATH.stat().st_size / (1024*1024):.1f} Mo")
else:
    print("↓ Téléchargement du DEM depuis Google Drive...")
    gdown.download(id=FILE_ID, output=str(DEM_PATH), quiet=False)
    print(f"✓ Téléchargé → {DEM_PATH}")

# -----------------------------------------------------------------------------
# 2. Vérification de l'intégrité du raster
# -----------------------------------------------------------------------------
print("\n→ Vérification du raster...")
try:
    with rasterio.open(DEM_PATH) as src:
        print(f"  CRS              : {src.crs}")
        print(f"  Dimensions       : {src.width} × {src.height}")
        print(f"  Résolution       : {src.res}")
        print(f"  Nombre de bandes : {src.count}")
        print(f"  Type de données  : {src.dtypes[0]}")
        print(f"  Emprise          : {src.bounds}")
    print("✓ DEM valide et lisible")
except Exception as e:
    print(f"❌ Erreur de lecture du DEM : {e}")

## 4. Vérification globale des données brutes

Cette section contrôle que toutes les sources sont bien présentes et lisibles.

In [ ]:
from pathlib import Path

print("=" * 70)
print("RÉSUMÉ DES DONNÉES BRUTES TÉLÉCHARGÉES")
print("=" * 70)

# OSM
osm_dir = Path("donnees_brutes/osm")
print(f"\n1. OpenStreetMap (Geofabrik)")
print(f"   Dossier : {osm_dir}")
print(f"   Fichiers .shp trouvés : {len(list(osm_dir.glob('*.shp')))}")

# Hydro
hydro_dir = Path("donnees_brutes/source2/extracted/hydro-maroc(CartoMaroc)")
print(f"\n2. Hydrographie CartoMaroc")
print(f"   Dossier : {hydro_dir}")
if hydro_dir.exists():
    print(f"   Fichiers .shp trouvés : {len(list(hydro_dir.glob('*.shp')))}")
else:
    print("   ⚠ Dossier non trouvé")

# DEM
dem_path = Path("donnees_brutes/dem/DEM_Grand_Casablanca_30m.tif")
print(f"\n3. DEM Grand Casablanca 30 m")
print(f"   Fichier : {dem_path}")
print(f"   Présent : {dem_path.exists()}")
if dem_path.exists():
    print(f"   Taille : {dem_path.stat().st_size / (1024*1024):.1f} Mo")

print("\n" + "=" * 70)
print("✓ Vérification terminée")

## Fin de l’acquisition des données brutes

Les trois sources sont maintenant disponibles localement de manière reproductible.

**Prochaine étape** :  
→ Reprojection de toutes les couches vers `EPSG:26191`  
→ Création du fichier `SOURCES.md`  
→ Import dans PostGIS